# Archived Databricks example

Historical source and outputs, not locally executable. Nothing was run during conversion. Databricks services, datasets and original image assets may be unavailable; image references are preserved, not downloaded. Outputs and errors are archived, not new execution.

# COPY INTO and MERGE Commands

### COPY INTO Command

> - Incrementally loads data into Delta Lake tables from Cloud Storage  
> - Supports schema evolution  
> - Supports wide range of file formats (CSV, JSON, Parquet, Delta)  
> - Alternative to Auto Loader for batch ingestion  

Documentation - [COPY INTO](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/delta-copy-into)

#### Create the table to copy the data into

Archived Databricks source — shown for reading, not Python execution.

```sql
CREATE TABLE IF NOT EXISTS demo.delta_lake.raw_stock_prices;
```

In [ ]:
# Archived output only; no computation.

Archived table (empty):

#### Incrementally load new files into the table

Archived Databricks source — shown for reading, not Python execution.

```sql
DELETE FROM demo.delta_lake.raw_stock_prices;

COPY INTO demo.delta_lake.raw_stock_prices
FROM 'abfss://demo@deacourseextdl.dfs.core.windows.net/landing/stock_prices'
FILEFORMAT = JSON
FORMAT_OPTIONS ('inferSchema' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true');
```

In [ ]:
# Archived output only; no computation.

Archived table:
num_affected_rows	num_inserted_rows	num_skipped_corrupt_files
6	6	0

Historical archived error:
{
  "baseErrorDetails": {
    "addedWidgets": {},
    "arguments": {},
    "datasetInfos": [],
    "jupyterProps": null,
    "metadata": {},
    "removedWidgets": [],
    "sqlProps": {
      "breakingChangeInfo": null,
      "errorClass": "PARSE_SYNTAX_ERROR",
      "pysparkCallSite": null,
      "pysparkFragment": null,
      "pysparkSummary": null,
      "sqlState": "42601",
      "stackTrace": null,
      "startIndex": null,
      "stopIndex": null
    },
    "stackFrames": [
      "org.apache.spark.sql.catalyst.parser.ParseException: \n[PARSE_SYNTAX_ERROR] Syntax error at or near ','. SQLSTATE: 42601 (line 4, pos 29)\n\n== SQL ==\nCOPY INTO demo.delta_lake.raw_stock_prices\nFROM 'abfss:REDACTED_LOCAL_PART@deacourseextdl.dfs.core.windows.net/landing/stock_prices'\nFILEFORMAT = JSON\nFORMAT_OPTIONS ('inferSchema', 'true')\n-----------------------------^^^\nCOPY_OPTIONS ('mergeSchema' = 'true')\n\n\tat org.apache.spark.sql.catalyst.parser.ParseException.with

Archived Databricks source — shown for reading, not Python execution.

```sql
SELECT * FROM demo.delta_lake.raw_stock_prices;
```

In [ ]:
# Archived output only; no computation.

Archived table:
price	status	stock_id	trading_date
233.5	ACTIVE	AAPL	2025-02-12
2805.0	ACTIVE	GOOGL	2025-02-12
335.0	ACTIVE	MSFT	2025-02-12
3520.0	ACTIVE	AMZN	2025-02-12
405.0	ACTIVE	META	2025-02-12
None	DELISTED	TSLA	2025-02-12

### MERGE Statement
> - Used for upserts (Insert/ Update/ Delete operations in a single statement)
> - Allows merging new data into a target table based on matching condition

Documentation - [MERGE INTO](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/delta-merge-into)

#### Create the table to merge the data into

Archived Databricks source — shown for reading, not Python execution.

```sql
CREATE TABLE IF NOT EXISTS demo.delta_lake.stock_prices
(
  stock_id STRING,
  price DOUBLE,
  trading_date DATE
);
```

In [ ]:
# Archived output only; no computation.

Archived table (empty):

#### Merge the source data into target table
1. Insert new stocks received
2. Update price and trading_date if updates receieved
3. Delete stocks which are de-listed from the exchange (status = 'DELISTED')

Archived Databricks source — shown for reading, not Python execution.

```sql
MERGE INTO demo.delta_lake.stock_prices AS target
USING demo.delta_lake.raw_stock_prices AS source
   ON target.stock_id = source.stock_id
WHEN MATCHED AND source.status = 'ACTIVE' THEN
    UPDATE SET target.price = source.price, target.trading_date = source.trading_date
WHEN MATCHED AND source.status = 'DELISTED' THEN    
    DELETE
WHEN NOT MATCHED AND source.status ='ACTIVE' THEN     
    INSERT (stock_id, price, trading_date) VALUES (source.stock_id, source.price, source.trading_date);
```

In [ ]:
# Archived output only; no computation.

Archived table:
num_affected_rows	num_updated_rows	num_deleted_rows	num_inserted_rows
6	4	1	1

Archived Databricks source — shown for reading, not Python execution.

```sql
SELECT * FROM demo.delta_lake.stock_prices;
```

In [ ]:
# Archived output only; no computation.

Archived table:
stock_id	price	trading_date
AAPL	233.5	2025-02-12
GOOGL	2805.0	2025-02-12
MSFT	335.0	2025-02-12
AMZN	3520.0	2025-02-12
META	405.0	2025-02-12